In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("cscie103_catalog.final_project.sales")

display(df.limit(10))

In [0]:
w = Window.partitionBy("store_nbr", "product_family").orderBy("date")

lags = [1, 7, 14, 28, 30, 60, 90]

for l in lags:
    df = df.withColumn(f"sales_lag_{l}", F.lag("sales", l).over(w))

df = df.withColumn("rolling_mean_7",  F.avg("sales").over(w.rowsBetween(-6, 0)))
df = df.withColumn("rolling_mean_28", F.avg("sales").over(w.rowsBetween(-27, 0)))
df = df.withColumn("rolling_mean_90", F.avg("sales").over(w.rowsBetween(-89, 0)))

df = df.withColumn("rolling_std_7",  F.stddev("sales").over(w.rowsBetween(-6, 0)))
df = df.withColumn("rolling_std_28", F.stddev("sales").over(w.rowsBetween(-27, 0)))

df = df.withColumn("day_of_week", F.dayofweek("date"))
df = df.withColumn("day_of_month", F.dayofmonth("date"))
df = df.withColumn("is_weekend", F.col("day_of_week").isin([1,7]).cast("int"))
df = df.withColumn("is_month_start", (F.dayofmonth("date") == 1).cast("int"))
df = df.withColumn("is_month_end", (F.last_day("date") == F.col("date")).cast("int"))
df = df.withColumn("is_payday", F.col("day_of_month").isin([15, 30, 31]).cast("int"))

In [0]:
store_type_freq = (
    df.groupBy("store_type")
      .count()
      .withColumnRenamed("count", "store_type_freq")
)

df = df.join(store_type_freq, on="store_type", how="left")

product_family_freq = (
    df.groupBy("product_family")
      .count()
      .withColumnRenamed("count", "product_family_freq")
)

df = df.join(product_family_freq, on="product_family", how="left")

In [0]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_counts)


In [0]:
# Drop rows where lag features are null
df = df.dropna(subset=["sales_lag_90"])

In [0]:
df.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("cscie103_catalog.final_project.silver_ml_features")
